In [9]:
import os
from chess import pgn
from tqdm import tqdm
import numpy as np
from chess import Board
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
import time

files = [file for file in os.listdir("../data/pgn") if file.endswith(".pgn")]

# Dataset Initialization

In [10]:
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
    return games

LIMITS = min(len(files), 24)
games = []
i = 1
for file in tqdm(files):
    games.extend(load_pgn(f"../data/pgn/{file}"))
    if (i >= LIMITS):
        break
    i += 1

 29%|██▉       | 23/79 [00:26<01:03,  1.14s/it]


# Neural Network

In [11]:
def board_to_mat(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix

def create_input_for_nn(games):
    x = []
    y = []
    for game in games:
        board = game.board()
        for move in game.mainline_moves():
            x.append(board_to_mat(board))
            y.append(move.uci())
            board.push(move)
        return x, y

def encode_moves(moves):
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int

x, y = create_input_for_nn(games)
y, move_to_int = encode_moves(y)
y = to_categorical(y, num_classes=len(move_to_int))
x = np.array(x)              

In [13]:
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(8, 8, 12)),
    Conv2D(128, (3,3), activation='relu'),
    Flatten(),
    Dense(256, activation='relu'),
    Dense(len(move_to_int), activation='softmax')
])
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(x, y, epochs=100, validation_split=0.1, batch_size=64)
model.save("../models/TF_100Epochs.keras")

c:\Users\bhumi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 6, 6, 64)       │         6,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 56)             │        14,392 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 619,768 (2.36 MB)

 Trainable params: 619,768 (2.36 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0196 - loss: 4.0241 - val_accuracy: 0.0000e+00 - val_loss: 4.0514
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.1373 - loss: 3.9479 - val_accuracy: 0.0000e+00 - val_loss: 4.1008
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.1961 - loss: 3.8716 - val_accuracy: 0.0000e+00 - val_loss: 4.1693
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.1765 - loss: 3.7790 - val_accuracy: 0.0000e+00 - val_loss: 4.2683
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.2157 - loss: 3.6697 - val_accuracy: 0.0000e+00 - val_loss: 4.3982
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.2353 - loss: 3.5462 - val_accuracy: 0.0000e+00 - val_loss: 4.5560
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.2549 - loss: 3.4104 - val_accuracy: 0.0000e+00 - val_loss: 4.7375
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.2941 - loss: 3.2593 - val_ac